# 01 · The class-balanced dataset (`Dataset_balanced`)

`Dataset_balanced` is built from the original 41-slide rat dataset by `scripts/build_dataset.py`.
In each task and in each of train, validation and test **separately**, every class is cut
to the size of that split's smallest class; nothing is oversampled or augmented. Slides
never change split, every slide keeps patches (spread evenly along its tile grid), and
validation is the exact set of animals VMLevaluator holds out at run time (20% of animals
per class, seed 95).

This notebook re-runs the build's verification and prints the patch and slide counts.
With the original data these are the counts quoted in Methods and Table 1. On the
synthetic data, the same build turns the synthetic raw `Dataset` (imbalanced, in the
original's folder numbering) into a balanced `Dataset_balanced`. To build by hand:
`python scripts/build_dataset.py --only Dataset_balanced` (it skips a dataset folder that exists).

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
# The histology images are not distributed. Without them this notebook runs on the
# synthetic dataset (scripts/make_synthetic_data.py) and writes to runs/synthetic/.
SYNTHETIC = (os.environ.get("VML_SYNTHETIC") == "1"          # forced, or no original data
             or not ((HERE / "data" / "Dataset_balanced" / "dataset.yaml").is_file() or "VML_DATA" in os.environ))
if SYNTHETIC:
    os.environ["VML_SYNTHETIC"] = "1"
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
if SYNTHETIC:   # generate and balance the synthetic data once (both steps skip existing output)
    subprocess.run([PY, str(HERE / "scripts" / "make_synthetic_data.py"), "--out", str(P.DATASETS)], check=True)
    subprocess.run([PY, str(HERE / "scripts" / "build_dataset.py"), "--only", "Dataset_balanced"],
                   check=True, capture_output=True)
print("mode           :", "SYNTHETIC data (not the manuscript's)" if SYNTHETIC else "original data")
print("dataset        :", P.DATA)
print("human data     :", P.HUMAN_ROOT)
print("results (used) :", P.RESULTS)

In [ ]:
r = subprocess.run([sys.executable, str(HERE / "scripts" / "build_dataset.py"), "--check", "--only", "Dataset_balanced"],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "dataset verification failed" 

In [ ]:
rep = json.loads(P.BUILD_REPORT.read_text())["Dataset_balanced"]
rows = []
for task, t in rep["tasks"].items():
    s = t["splits"]
    k = len(t["class_names"])
    rows.append(dict(task=task, classes=" / ".join(t["class_names"]),
                     patches_per_class_fit=s["train"]["per_class"], patches_per_class_val=s["val"]["per_class"],
                     patches_per_class_test=s["test"]["per_class"],
                     train_patches=k * (s["train"]["per_class"] + s["val"]["per_class"]), test_patches=k * s["test"]["per_class"],
                     train_wsis=sum(s["train"]["slides"].values()) + sum(s["val"]["slides"].values()),
                     test_wsis=sum(s["test"]["slides"].values()),
                     train_wsis_per_class="/".join(str(s["train"]["slides"][c] + s["val"]["slides"][c]) for c in t["class_names"]),
                     test_wsis_per_class="/".join(str(s["test"]["slides"][c]) for c in t["class_names"])))
counts = pd.DataFrame(rows)
counts

Methods states, for the original data: Control vs VML 20 / 8 WSIs and 3,428 / 1,466
patches; Poor vs Very poor 11 / 6 WSIs and 3,344 / 1,800 patches; three-class 15 / 11 WSIs
(4/6/5 and 5/4/2 per class) and 2,985 / 1,329 patches. Training patches include the
validation slides. These counts are also frozen in `reference/dataset_report.json`.

In [ ]:
expected = {"control_vml": (20, 8, 3428, 1466), "poor_very_poor": (11, 6, 3344, 1800), "multiclass": (15, 11, 2985, 1329)}
if SYNTHETIC:
    print("Synthetic data: the counts above are the synthetic dataset's. The manuscript's, from reference/dataset_report.json:")
    ref = json.loads((P.REFERENCE / "dataset_report.json").read_text())["Dataset_balanced"]["tasks"]
    for t, (tr, te, ptr, pte) in expected.items():
        s = ref[t]["splits"]; k = len(ref[t]["class_names"])
        got = (sum(s["train"]["slides"].values()) + sum(s["val"]["slides"].values()), sum(s["test"]["slides"].values()),
               k * (s["train"]["per_class"] + s["val"]["per_class"]), k * s["test"]["per_class"])
        print(f"  {t:15s} {got}  {'matches Methods' if got == (tr, te, ptr, pte) else 'DIFFERS'}")
else:
    for _, r in counts.iterrows():
        got = (r.train_wsis, r.test_wsis, r.train_patches, r.test_patches)
        print(f"{r.task:15s} {got}  {'matches Methods' if got == expected[r.task] else 'DIFFERS from ' + str(expected[r.task])}")